In [11]:
# ==========================================
# PARCHE ANTI-FIREWALL / PROXY CORPORATIVO
# ==========================================
import urllib3
from urllib3.connectionpool import HTTPSConnectionPool
import os

# 1. Apagamos la verificación nativa de Python
os.environ["PYTHONHTTPSVERIFY"] = "0"
os.environ["REQUESTS_CA_BUNDLE"] = ""

# 2. Silenciamos las alertas rojas para que no ensucien la pantalla
urllib3.disable_warnings()

# 3. Forzamos a urllib3 (el motor que usa Pinecone) a ignorar certificados
orig_init = HTTPSConnectionPool.__init__
def patched_init(self, *args, **kwargs):
    kwargs['cert_reqs'] = 'CERT_NONE'
    kwargs['assert_hostname'] = False
    orig_init(self, *args, **kwargs)
HTTPSConnectionPool.__init__ = patched_init

In [12]:
# ==========================================
# PARCHE ANTI-FIREWALL (A PRUEBA DE REINICIOS)
# ==========================================
import urllib3
from urllib3.connectionpool import HTTPSConnectionPool
import httpx

os.environ["PYTHONHTTPSVERIFY"] = "0"
os.environ["REQUESTS_CA_BUNDLE"] = ""
urllib3.disable_warnings()

# El candado: Si ya está parchado, no lo vuelve a parchar
if not hasattr(HTTPSConnectionPool, "_ya_parchado"):
    orig_init = HTTPSConnectionPool.__init__
    def patched_init(self, *args, **kwargs):
        kwargs['cert_reqs'] = 'CERT_NONE'
        kwargs['assert_hostname'] = False
        orig_init(self, *args, **kwargs)
    HTTPSConnectionPool.__init__ = patched_init
    HTTPSConnectionPool._ya_parchado = True # Cerramos el candado

# Cliente para OpenAI
cliente_sin_ssl = httpx.Client(verify=False)

In [13]:
# ==========================================
# 1. SETUP E IMPORTACIONES
# ==========================================
# Importamos la clase principal para crear el mapa del grafo
from langgraph.graph import StateGraph
# Importamos RunnableLambda para convertir funciones normales en nodos compatibles con el grafo
from langchain_core.runnables import RunnableLambda
# Importamos el modelo de IA que va a razonar (Cerebro)
from langchain_openai import ChatOpenAI
# Importamos el molde para armar los prompts conversacionales
from langchain_core.prompts import ChatPromptTemplate
# Importamos la herramienta gratuita para buscar en internet (DuckDuckGo)
from ddgs import DDGS
# Importamos TypedDict para tipar fuertemente nuestro estado (memoria)
from typing import TypedDict
# Librería para renderizar el dibujo del grafo al final
import os
import yaml
from langgraph.graph import StateGraph, END
from langchain_core.output_parsers import StrOutputParser

contenido_RAG = "casos clínicos, sintomas, diagnosticos y formas de diagnostico"
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]
os.environ["PINECONE_API_KEY"] = config["PINECONE_API_KEY"]


class State(TypedDict):
    """Definimos la estructura de nuestro estado (memoria) usando TypedDict para tipado fuerte."""
    pregunta: str
    contenidoInternet: str
    contenidoRAG: str
    historial: list 
    respuesta: str
    next_step: str


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:


from itertools import chain


def recibe_pregunta(state: State) -> State:

    if not state.get("historial"):
        state["historial"] = []
    
    state["historial"].append(f"Pregunta: {state['pregunta']}")
    
    return state

def decision(state: State) -> str:
    

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", '''Eres un asistente inteligente que decide la mejor estrategia para responder a una pregunta.
        Tienes tres opciones: buscar en internet, buscar en una base de datos RAG o generar una respuesta directa utilizando un modelo de IA. 
        Basas tu decisión en la pregunta recibida.
        En nuestro RAG tenemos información sobre {explicacion_contenido_RAG}, pero no sobre cultura pop o eventos actuales.
        Responde ÚNICAMENTE con una de estas tres opciones: 
         "buscar_en_internet" si es una pregunta que no está en el RAG, 
         "buscar_en_rag" si el contenido puede estar en el RAG contenido del RAG ({explicacion_contenido_RAG}) o 
         "consultar_llm" si no es una pregunta que tiene que ver con el contenido del RAG ni es algo para buscar en internet.
          No agregues puntos, ni texto extra.'''),
        ("assistant", "Historial de la conversación hasta ahora: {historial}"),
        ("user", "Pregunta: {pregunta}")
        # Y aquí le podemos dar contexto de la conversación, historial, etc, para que tome una mejor decisión
    ])
    
    chain = chat_prompt | llm | StrOutputParser() 

    pregunta_usuario = state["pregunta"]

    respuesta = chain.invoke({
        "pregunta": state["pregunta"], 
        "explicacion_contenido_RAG": contenido_RAG,
        "historial": "\n".join(state.get("historial", "No hay historial previo hasta ahora"))
    })

    return {"next_step": respuesta.strip('"')}

def buscar_en_rag(state: State) -> State:
    from pinecone import Pinecone
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings

    pinecone_client = Pinecone()

    cliente_sin_ssl = httpx.Client(verify=False)

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small", 
        http_client=cliente_sin_ssl
    )

    query_vector = embeddings.embed_query(state["pregunta"])

    indice = PineconeVectorStore(
        pinecone_client=pinecone_client,
        index_name="rag-casos-clinicos",
        text_key="text",
        metadata_keys=["text"]
    )
    
    # pasar pregunta del usuario a vector embedding
    respuesta_pinecone = indice.query(
    vector=query_vector, 
    top_k=3,                # Devuelve solo los 3 resultados más similares (los "vecinos más cercanos")
    include_metadata=True
    )

    documentos_recuperados = respuesta_pinecone["matches"]

    state["contenidoRAG"] = "\n".join([doc["metadata"]["text"] for doc in documentos_recuperados])
    
    return state

def buscar_en_internet(state: State) -> State:
    query = state["pregunta"]
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
    contenido_internet = "\n".join([result["title"] + ": " + result["href"] for result in results]) 
    state["contenidoInternet"] = contenido_internet
    return state


def consultar_llm(state: State) -> State:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente inteligente que responde preguntas utilizando tu conocimiento previo."),
        ("user", "Pregunta: {pregunta}"),
        ("assistant", "Historial de la conversación hasta ahora: {historial}")

    ])
    
    chain = prompt_template | llm | StrOutputParser()

    respuesta = chain.invoke({"pregunta": state["pregunta"], "historial": "\n".join(state["historial"])})

    state["respuesta"] = respuesta
    
    return state

def sintetizar_respuesta(state: State) -> State:
    
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente inteligente que sintetiza respuestas finales a partir de toda la información disponible."),
        ("user", "Pregunta: {pregunta}\nContexto RAG: {contenidoRAG}\nContexto Internet: {contenidoInternet}\nHistorial: {historial}")
    ])
    
    chain = prompt_template | llm | StrOutputParser()
    
    respuesta = chain.invoke({
        "pregunta": state["pregunta"],
        "contenidoRAG": state.get("contenidoRAG", ""),
        "contenidoInternet": state.get("contenidoInternet", ""),
        "historial": "\n".join(state["historial"])
    })
    
    state["respuesta"] = respuesta

    return state


def responder(state: State) -> State:

    respuesta_final = state.get("respuesta", "Sin respuesta")
    state["historial"].append(f"Respuesta: {respuesta_final}")
    return state

In [ ]:
graph = StateGraph(State)
graph.add_node("recibe_pregunta", RunnableLambda(recibe_pregunta))
graph.add_node("decision", RunnableLambda(decision))
graph.add_node("buscar_en_internet", RunnableLambda(buscar_en_internet))
graph.add_node("buscar_en_rag", RunnableLambda(buscar_en_rag))
graph.add_node("consultar_llm", RunnableLambda(consultar_llm))
graph.add_node("sintetizar_respuesta", RunnableLambda(sintetizar_respuesta))
graph.add_node("responder", RunnableLambda(responder))

In [ ]:
graph.set_entry_point("recibe_pregunta")
graph.add_edge("recibe_pregunta", "decision")
graph.add_conditional_edges(
    "decision",
lambda state: state["next_step"], # variable temporal que contiene la respuesta del LLM indicando la siguiente acción a tomar
    {   "consultar_llm": "consultar_llm",
        "buscar_en_internet": "buscar_en_internet",
        "buscar_en_rag": "buscar_en_rag",
    }
)
graph.add_edge("consultar_llm", "responder")
graph.add_edge("buscar_en_internet", "sintetizar_respuesta")
graph.add_edge("buscar_en_rag", "sintetizar_respuesta")
graph.add_edge("sintetizar_respuesta", "responder")
graph.add_edge("responder", END)


In [ ]:
ejecutable = graph.compile()

In [ ]:
estado_inicial = {
    "pregunta": "Hola"
}

resultado_final = ejecutable.invoke(estado_inicial)

print("PREGUNTA: " + estado_inicial["pregunta"])
print("RESPUESTA: " + resultado_final["respuesta"])



C:\Users\320258215\AppData\Local\Temp\ipykernel_31000\3622708102.py:78: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().search(query, max_results=3))


AttributeError: 'DDGS' object has no attribute 'search'